## Read from bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
df.display()

### Fix invalid dates, sales and price

In [0]:
query = """
  SELECT
    sls_ord_num,
    sls_prd_key,
    sls_cust_id,
    CASE 
      WHEN sls_order_dt <= 0 OR LENGTH(sls_order_dt) != 8 THEN NULL
      ELSE to_date(sls_order_dt::STRING, 'yyyyMMdd')
    END AS sls_order_dt,
    CASE 
      WHEN sls_ship_dt <= 0 OR LENGTH(sls_ship_dt) != 8 THEN NULL
      ELSE to_date(sls_ship_dt::STRING, 'yyyyMMdd')
    END AS sls_ship_dt,
    CASE 
      WHEN sls_due_dt <= 0 OR LENGTH(sls_due_dt) != 8 THEN NULL
      ELSE to_date(sls_due_dt::STRING, 'yyyyMMdd')
    END AS sls_due_dt,
    CASE 
      WHEN sls_sales <= 0 OR sls_sales IS NULL OR sls_sales != sls_quantity * sls_price
        THEN sls_quantity * ABS(sls_price)
      ELSE sls_sales
    END AS sls_sales,
    sls_quantity,
    CASE 
      WHEN sls_price = 0 OR sls_price IS NULL
        THEN sls_sales/NULLIF(sls_quantity,0)
      ELSE ABS(sls_price)
    END AS sls_price
  FROM workspace.bronze.crm_sales_details
"""

df = spark.sql(query)
df.display()

### Rename table schema

In [0]:
rename_shcema = {
    "sls_ord_num": "sls_order_number",
    "sls_prd_key": "sls_product_key",
    "sls_cust_id": "sls_customer_id",
    "sls_order_dt": "sls_order_date",
    "sls_ship_dt": "sls_shipping_date",
    "sls_due_dt": "sls_due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old, new in rename_shcema.items():
    df = df.withColumnRenamed(old, new)
df.display()

### Load into silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silvercrm_sales_details")

In [0]:
display(spark.table("workspace.silver.crm_sales_details"))